# Multi-Dataset Multi-Task Plant Disease Retraining (Solution 1)
### Precision AgriTech Platform - In-Field Domain Shift Elimination

**Objective:** Retrain the disease diagnosis model with authentic in-field datasets (Mendeley Potato, Mendeley Corn, Mendeley Tomato, PlantDoc) alongside PlantVillage clean lab baselines. Overcomes domain-shift shortcut learning on soil, shadows, weeds, and complex agricultural backgrounds using a **Dual-Head MobileNetV3-Large** architecture.

## 1. Hardware Verification & Environment Setup

In [ ]:
!nvidia-smi
import random, os, json, time
import numpy as np
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__} | Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

## 2. Dependencies & Colab Workspace Setup

In [ ]:
%pip install -q torch torchvision torchaudio scikit-learn pillow matplotlib onnx onnxruntime

# Optional: Mount Google Drive or set dataset root
# from google.colab import drive
# drive.mount('/content/drive')
DATASET_ROOT = None  # Set to Path('/content/dataset') if remapping dataset folders

## 3. Architecture Definition: Dual-Head MobileNetV3-Large

In [ ]:
from typing import Tuple, Dict, Any, List, Optional
import torch.nn as nn
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights
from sklearn.metrics import f1_score, accuracy_score

class MultiTaskPlantClassifier(nn.Module):
    def __init__(self, num_crops: int = 4, num_diseases: int = 8, pretrained: bool = True, dropout_rate: float = 0.2):
        super().__init__()
        weights = MobileNet_V3_Large_Weights.DEFAULT if pretrained else None
        base_model = mobilenet_v3_large(weights=weights)
        self.features = base_model.features
        self.avgpool = base_model.avgpool
        in_features = getattr(base_model.classifier[0], "in_features", 960)

        # Head 1: Crop Species (Macro-morphology: 4 classes)
        self.crop_head = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.Hardswish(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(256, num_crops),
        )
        # Head 2: Disease Pathology (Micro-lesions: 8 classes)
        self.disease_head = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.Hardswish(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(512, num_diseases),
        )

    def extract_features(self, x: torch.Tensor) -> torch.Tensor:
        feat = self.features(x)
        feat = self.avgpool(feat)
        return torch.flatten(feat, 1)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        feat = self.extract_features(x)
        return self.crop_head(feat), self.disease_head(feat)

class MultiTaskLoss(nn.Module):
    def __init__(self, weight_crop: float = 0.4, weight_disease: float = 0.6, label_smoothing: float = 0.1):
        super().__init__()
        self.weight_crop = weight_crop
        self.weight_disease = weight_disease
        self.criterion_crop = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
        self.criterion_disease = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    def forward(self, crop_logits, disease_logits, crop_targets, disease_targets):
        l_crop = self.criterion_crop(crop_logits, crop_targets)
        l_dis = self.criterion_disease(disease_logits, disease_targets)
        total = (self.weight_crop * l_crop) + (self.weight_disease * l_dis)
        return total, {"loss_crop": l_crop.detach(), "loss_disease": l_dis.detach(), "total_loss": total.detach()}

## 4. Dataset Loading & Cross-Platform In-Field Augmentations

In [ ]:
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.20), ratio=(0.3, 3.3), value="random"),
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class ColabMultiTaskDataset(Dataset):
    """Cross-platform dataset resolving relative or remapped file paths."""
    def __init__(self, manifest_path: str, transform=None, dataset_root: Optional[str] = None):
        with open(manifest_path, 'r', encoding='utf-8') as f:
            self.records = json.load(f)
        self.transform = transform
        self.dataset_root = Path(dataset_root) if dataset_root else None

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        raw_path = rec["file_path"].replace("\\", "/")
        if self.dataset_root:
            parts = raw_path.split("data/raw/")
            p = self.dataset_root / parts[-1] if len(parts) > 1 else self.dataset_root / Path(raw_path).name
        else:
            p = Path(raw_path)

        with Image.open(p) as img:
            img = img.convert("RGB")
            x = self.transform(img) if self.transform else transforms.ToTensor()(img)
        return x, torch.tensor(rec["crop_idx"], dtype=torch.long), torch.tensor(rec["disease_idx"], dtype=torch.long)

## 5. Training Loop with Linear Warmup, AdamW, & Cosine Annealing

In [ ]:
TRAIN_MANIFEST = "ml-retrain-experimental/data/manifests/train_manifest.json"
VAL_MANIFEST = "ml-retrain-experimental/data/manifests/val_manifest.json"

train_ds = ColabMultiTaskDataset(TRAIN_MANIFEST, transform=train_transform, dataset_root=DATASET_ROOT)
val_ds = ColabMultiTaskDataset(VAL_MANIFEST, transform=val_transform, dataset_root=DATASET_ROOT)

use_pin_memory = torch.cuda.is_available()
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=use_pin_memory)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=use_pin_memory)

model = MultiTaskPlantClassifier(num_crops=4, num_diseases=8, pretrained=True).to(device)
criterion = MultiTaskLoss(weight_crop=0.4, weight_disease=0.6, label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

epochs = 25
warmup_epochs = 3
warmup_scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs)
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(epochs - warmup_epochs, 1), eta_min=1e-6)
scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs])

best_joint_acc = 0.0
history = []

for epoch in range(1, epochs + 1):
    # Train
    model.train()
    train_loss = 0.0
    tr_cp, tr_ct, tr_dp, tr_dt = [], [], [], []
    for imgs, crops, dis in train_loader:
        imgs, crops, dis = imgs.to(device), crops.to(device), dis.to(device)
        optimizer.zero_grad()
        c_log, d_log = model(imgs)
        loss, _ = criterion(c_log, d_log, crops, dis)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item() * imgs.size(0)
        tr_cp.extend(torch.argmax(c_log, dim=-1).cpu().tolist())
        tr_ct.extend(crops.cpu().tolist())
        tr_dp.extend(torch.argmax(d_log, dim=-1).cpu().tolist())
        tr_dt.extend(dis.cpu().tolist())
    scheduler.step()

    tr_joint_acc = sum(1 for cp, ct, dp, dt in zip(tr_cp, tr_ct, tr_dp, tr_dt) if cp == ct and dp == dt) / len(tr_ct)

    # Validation
    model.eval()
    val_loss = 0.0
    val_cp, val_ct, val_dp, val_dt = [], [], [], []
    with torch.no_grad():
        for imgs, crops, dis in val_loader:
            imgs, crops, dis = imgs.to(device), crops.to(device), dis.to(device)
            c_log, d_log = model(imgs)
            loss, _ = criterion(c_log, d_log, crops, dis)
            val_loss += loss.item() * imgs.size(0)
            val_cp.extend(torch.argmax(c_log, dim=-1).cpu().tolist())
            val_ct.extend(crops.cpu().tolist())
            val_dp.extend(torch.argmax(d_log, dim=-1).cpu().tolist())
            val_dt.extend(dis.cpu().tolist())

    crop_acc = accuracy_score(val_ct, val_cp)
    disease_acc = accuracy_score(val_dt, val_dp)
    joint_acc = sum(1 for cp, ct, dp, dt in zip(val_cp, val_ct, val_dp, val_dt) if cp == ct and dp == dt) / len(val_ct)
    dis_f1 = f1_score(val_dt, val_dp, average="macro", zero_division=0)

    epoch_stats = {
        "epoch": epoch,
        "train_loss": train_loss / len(train_ds),
        "train_joint_acc": tr_joint_acc,
        "val_loss": val_loss / len(val_ds),
        "val_crop_acc": crop_acc,
        "val_disease_acc": disease_acc,
        "val_joint_acc": joint_acc,
        "val_disease_f1": dis_f1,
    }
    history.append(epoch_stats)

    print(f"Epoch {epoch:02d}/{epochs:02d} | Train Loss: {epoch_stats['train_loss']:.4f} (Joint: {tr_joint_acc*100:.1f}%) | Val Loss: {epoch_stats['val_loss']:.4f} | Val Joint Acc: {joint_acc*100:.2f}% | Dis F1: {dis_f1*100:.2f}%")
    if joint_acc > best_joint_acc:
        best_joint_acc = joint_acc
        ckpt = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_score": joint_acc,
            "metrics": epoch_stats,
        }
        torch.save(ckpt, "best_multitask_model.pth")
        print(f"  --> [SAVED] New best model checkpoint: Joint Acc = {joint_acc*100:.2f}%")

## 6. Training Curve Visualization

In [ ]:
import matplotlib.pyplot as plt

ep_list = [h["epoch"] for h in history]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(ep_list, [h["train_loss"] for h in history], label="Train Loss", color="tab:blue")
axes[0].plot(ep_list, [h["val_loss"] for h in history], label="Val Loss", color="tab:red", linestyle="--")
axes[0].set_title("Multi-Task Loss Curve")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True)

# Accuracy
axes[1].plot(ep_list, [h["train_joint_acc"]*100 for h in history], label="Train Joint Acc (%)", color="tab:blue")
axes[1].plot(ep_list, [h["val_joint_acc"]*100 for h in history], label="Val Joint Acc (%)", color="tab:green")
axes[1].plot(ep_list, [h["val_disease_f1"]*100 for h in history], label="Val Disease Macro F1 (%)", color="tab:orange", linestyle=":")
axes[1].set_title("Multi-Task Joint Accuracy & Macro F1")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Percentage (%)")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 7. Reload Best Checkpoint & Export to ONNX (with Parity Verification)

In [ ]:
# CRITICAL: Reload the best model checkpoint rather than exporting the final epoch
best_ckpt = torch.load("best_multitask_model.pth", map_location="cpu")
model.load_state_dict(best_ckpt["model_state_dict"])
model.eval()
model.cpu()
print(f"Loaded best model from Epoch {best_ckpt['epoch']} with Joint Accuracy: {best_ckpt['best_score']*100:.2f}%")

onnx_path = "agritech_multitask.onnx"
dummy_input = torch.randn(1, 3, 224, 224)

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["crop_logits", "disease_logits"],
    dynamic_axes={
        "input": {0: "batch_size"},
        "crop_logits": {0: "batch_size"},
        "disease_logits": {0: "batch_size"},
    },
    dynamo=False,
)
print(f"Successfully exported ONNX model to {onnx_path}")

# Verify with ONNX Runtime
import onnxruntime as ort
session = ort.InferenceSession(onnx_path)
ort_inputs = {"input": dummy_input.numpy()}
ort_crops, ort_dis = session.run(None, ort_inputs)

with torch.no_grad():
    pt_crops, pt_dis = model(dummy_input)

crop_diff = np.max(np.abs(ort_crops - pt_crops.numpy()))
dis_diff = np.max(np.abs(ort_dis - pt_dis.numpy()))
print(f"ONNX Runtime numerical parity verified! Max Crop diff: {crop_diff:.6f}, Max Disease diff: {dis_diff:.6f}")

## 8. Test Set Evaluation & Field Benchmark Reporting

In [ ]:
TEST_MANIFEST = "ml-retrain-experimental/data/manifests/test_manifest.json"

test_ds = ColabMultiTaskDataset(TEST_MANIFEST, transform=val_transform, dataset_root=DATASET_ROOT)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

model.eval()
model.to(device)

all_cp, all_ct, all_dp, all_dt = [], [], [], []
with torch.no_grad():
    for imgs, crops, dis in test_loader:
        imgs = imgs.to(device)
        c_log, d_log = model(imgs)
        all_cp.extend(torch.argmax(c_log, dim=-1).cpu().tolist())
        all_ct.extend(crops.tolist())
        all_dp.extend(torch.argmax(d_log, dim=-1).cpu().tolist())
        all_dt.extend(dis.tolist())

test_crop_acc = accuracy_score(all_ct, all_cp)
test_dis_acc = accuracy_score(all_dt, all_dp)
test_joint_acc = sum(1 for cp, ct, dp, dt in zip(all_cp, all_ct, all_dp, all_dt) if cp == ct and dp == dt) / len(all_ct)
test_dis_f1 = f1_score(all_dt, all_dp, average="macro", zero_division=0)

print("=" * 60)
print("FINAL HELD-OUT TEST BENCHMARK RESULTS")
print("=" * 60)
print(f"Total Test Samples Evaluated: {len(all_ct)}")
print(f"Crop Species Accuracy:       {test_crop_acc*100:.2f}%")
print(f"Disease Pathology Accuracy:  {test_dis_acc*100:.2f}%")
print(f"Joint Exact Match:           {test_joint_acc*100:.2f}%")
print(f"Disease Macro F1:            {test_dis_f1*100:.2f}%")
print("=" * 60)

# Download model weights and ONNX model directly in Colab:
# from google.colab import files
# files.download('best_multitask_model.pth')
# files.download('agritech_multitask.onnx')